In [1]:
!pip install -q google-cloud-aiplatform[evaluation] ipytest

In [2]:
import vertexai
from vertexai.generative_models import GenerativeModel
from vertexai.evaluation import EvalTask
import pandas as pd
import datetime
import ipytest
import pytest

ipytest.autoconfig()

PROJECT_ID = "qwiklabs-gcp-04-a13cec945fb9"
LOCATION = "us-east4"
MODEL_ID = "gemini-2.5-flash"

vertexai.init(project=PROJECT_ID, location=LOCATION)
model = GenerativeModel(MODEL_ID)

print(f"Initialized Vertex AI")
print(f"  Project : {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  Model   : {MODEL_ID}")

Initialized Vertex AI
  Project : qwiklabs-gcp-04-a13cec945fb9
  Location: us-east4
  Model   : gemini-2.5-flash


In [3]:
#had to add retry logic in here to keep from getting a 429
import time
import functools

def with_retry(fn, max_attempts=4, initial_delay=5):
    """Retry a function with exponential backoff on RESOURCE_EXHAUSTED errors."""
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        delay = initial_delay
        for attempt in range(max_attempts):
            try:
                return fn(*args, **kwargs)
            except Exception as e:
                if "RESOURCE_EXHAUSTED" in str(e) and attempt < max_attempts - 1:
                    print(f"[Rate limit] Waiting {delay}s before retry {attempt + 1}...")
                    time.sleep(delay)
                    delay *= 2
                else:
                    raise
    return wrapper


@with_retry
def classify_question(question: str) -> str:
    """
    Classify a citizen question into one of four government service categories.

    Returns:
        One of: 'Employment', 'General Information', 'Emergency Services', 'Tax Related'
    """
    response = model.generate_content(
        """You are a government service routing assistant.
Classify the citizen question below into exactly one of these four categories:
- Employment
- General Information
- Emergency Services
- Tax Related

Rules:
1. Output ONLY the category name, nothing else.
2. Do not include punctuation, explanation, or extra words.
3. Choose the single most relevant category.

Examples:
Question: How do I apply for unemployment benefits?
Output: Employment

Question: What are the city hall office hours?
Output: General Information

Question: There is a fire in my neighbor's house, who do I call?
Output: Emergency Services

Question: When is the deadline to file my property tax appeal?
Output: Tax Related

Question: {0}
Output: """.format(question)
    )
    return response.text.strip()


# Smoke test
test_questions = [
    "How do I file for unemployment after being laid off?",
    "Where is the nearest DMV office?",
    "There is a gas leak on my street, who do I contact?",
    "How do I get a copy of my W-2 from the city?",
]

print("Smoke test results:")
print("-" * 50)
for q in test_questions:
    category = classify_question(q)
    print(f"Q: {q}")
    print(f"   → {category}")
    print()

Smoke test results:
--------------------------------------------------
Q: How do I file for unemployment after being laid off?
   → Employment

Q: Where is the nearest DMV office?
   → General Information

Q: There is a gas leak on my street, who do I contact?
   → Emergency Services

Q: How do I get a copy of my W-2 from the city?
   → Tax Related



In [4]:
@with_retry
def generate_social_post(announcement: str) -> str:
    """
    Generate a government social media post for a public announcement.

    Returns:
        A social media post as a string.
    """
    response = model.generate_content(
        """You write social media posts for a government communications department.

Rules:
1. Keep posts under 280 characters.
2. Use a professional but approachable tone.
3. Include the hashtag #CityAlert at the end of every post.
4. Include one relevant emoji at the start of the post.
5. For emergencies, begin with ALERT:.

Examples:
Input: City Hall will be closed on Monday for Memorial Day.
Output: 🏛️ City Hall will be CLOSED Monday, May 27 in observance of Memorial Day. Regular hours resume Tuesday. #CityAlert

Input: A water main break on Oak Street may affect service to nearby residents.
Output: 🚨 ALERT: Water main break on Oak St. may affect service to nearby residents. Crews are on site. Updates to follow. #CityAlert

Input: {0}
Output: """.format(announcement)
    )
    return response.text.strip()


# Smoke test
test_announcements = [
    "All public schools will be closed tomorrow due to a winter storm warning.",
    "The downtown farmers market will be held Saturday from 8am to 1pm.",
    "A tornado warning has been issued for the county until 6pm tonight.",
]

print("Smoke test results:")
print("-" * 50)
for ann in test_announcements:
    post = generate_social_post(ann)
    print(f"Announcement: {ann}")
    print(f"Post ({len(post)} chars): {post}")
    print()

Smoke test results:
--------------------------------------------------
Announcement: All public schools will be closed tomorrow due to a winter storm warning.
Post (101 chars): 🚨 ALERT: All public schools closed tomorrow due to winter storm warning. Stay safe & warm! #CityAlert

Announcement: The downtown farmers market will be held Saturday from 8am to 1pm.
Post (142 chars): 🥕 Don't miss the Downtown Farmers Market this Saturday from 8 AM - 1 PM! Support local vendors & find fresh produce. See you there! #CityAlert

Announcement: A tornado warning has been issued for the county until 6pm tonight.
Post (154 chars): 🚨 ALERT: Tornado Warning issued for our county until 6 PM tonight. Seek shelter immediately in an interior room on the lowest floor. Stay safe! #CityAlert



In [5]:
@with_retry
def does_post_follow_rules(post: str) -> str:
    """
    Use the LLM to evaluate whether a social media post follows the generation rules.

    Returns:
        'Yes' if the post follows all rules, 'No' otherwise.
    """
    response = model.generate_content(
        """Does the following social media post follow ALL of these rules?
1. Under 280 characters in total length.
2. Ends with the hashtag #CityAlert.
3. Starts with a single emoji.

Output ONLY Yes or No.

Post: {0}
Output: """.format(post)
    )
    return response.text.strip()

In [6]:
%%ipytest -v

import time

@pytest.fixture(autouse=True)
def rate_limit_pause():
    yield
    time.sleep(3)  # 3s between each test to avoid rate limiting

class TestClassifyQuestion:

    # --- Employment ---
    def test_classify_unemployment(self):
        result = classify_question("How do I apply for unemployment benefits?")
        assert result == "Employment"

    def test_classify_job_posting(self):
        result = classify_question("Where can I find government job openings in the city?")
        assert result == "Employment"

    def test_classify_work_permit(self):
        result = classify_question("I need a work permit for my new job, how do I get one?")
        assert result == "Employment"

    # --- General Information ---
    def test_classify_office_hours(self):
        result = classify_question("What are the operating hours for the DMV?")
        assert result == "General Information"

    def test_classify_location(self):
        result = classify_question("Where is the nearest public library?")
        assert result == "General Information"

    def test_classify_general_procedure(self):
        result = classify_question("How do I renew my driver's license?")
        assert result == "General Information"

    # --- Emergency Services ---
    def test_classify_fire(self):
        result = classify_question("There is a fire in my neighbor's house, who do I call?")
        assert result == "Emergency Services"

    def test_classify_medical(self):
        result = classify_question("Someone collapsed on the street and is unresponsive.")
        assert result == "Emergency Services"

    def test_classify_gas_leak(self):
        result = classify_question("I smell gas in my building. What should I do?")
        assert result == "Emergency Services"

    # --- Tax Related ---
    def test_classify_property_tax(self):
        result = classify_question("When is the deadline to pay my property taxes?")
        assert result == "Tax Related"

    def test_classify_tax_refund(self):
        result = classify_question("How do I check the status of my city tax refund?")
        assert result == "Tax Related"

    def test_classify_tax_forms(self):
        result = classify_question("Where can I download the local income tax filing forms?")
        assert result == "Tax Related"

======================================= test session starts ========================================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 12 items

t_877af68422cd4f6fb17e1d829792dc42.py ............                                           [100%]

======================================= 12 passed in 48.19s ========================================


In [7]:
%%ipytest -v

@pytest.fixture(autouse=True)
def rate_limit_pause():
    yield
    time.sleep(3)  # 3s between each test to avoid rate limiting

class TestGenerateSocialPost:

    def test_school_closure_follows_rules(self):
        post = generate_social_post(
            "All public schools will be closed tomorrow due to a winter storm warning."
        )
        result = does_post_follow_rules(post)
        assert result == "Yes", f"Post did not follow rules: {post}"

    def test_holiday_closure_follows_rules(self):
        post = generate_social_post(
            "City Hall will be closed on July 4th for Independence Day."
        )
        result = does_post_follow_rules(post)
        assert result == "Yes", f"Post did not follow rules: {post}"

    def test_weather_emergency_follows_rules(self):
        post = generate_social_post(
            "A tornado warning has been issued for the county until 6pm tonight."
        )
        result = does_post_follow_rules(post)
        assert result == "Yes", f"Post did not follow rules: {post}"

    def test_road_closure_follows_rules(self):
        post = generate_social_post(
            "Main Street will be closed for repaving from Monday through Wednesday."
        )
        result = does_post_follow_rules(post)
        assert result == "Yes", f"Post did not follow rules: {post}"

    def test_community_event_follows_rules(self):
        post = generate_social_post(
            "The downtown farmers market returns this Saturday from 8am to 1pm."
        )
        result = does_post_follow_rules(post)
        assert result == "Yes", f"Post did not follow rules: {post}"

    def test_post_under_280_chars(self):
        """Direct assertion — post length must be under 280 characters."""
        post = generate_social_post(
            "Boil water advisory issued for the Riverside district due to a water main break."
        )
        assert len(post) <= 280, f"Post exceeds 280 chars ({len(post)}): {post}"

    def test_post_contains_hashtag(self):
        """Direct assertion — post must contain #CityAlert."""
        post = generate_social_post(
            "Free flu shots are available at the community center this weekend."
        )
        assert "#CityAlert" in post, f"Post missing #CityAlert: {post}"

    def test_post_does_not_follow_rules_detection(self):
        """Verify rule checker correctly identifies a non-compliant post."""
        bad_post = "City Hall closed Monday for the holiday. See you Tuesday!"
        result = does_post_follow_rules(bad_post)
        assert result == "No", f"Expected 'No' for non-compliant post, got: {result}"

======================================= test session starts ========================================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 8 items

t_877af68422cd4f6fb17e1d829792dc42.py ........                                               [100%]

=================================== 8 passed in 73.56s (0:01:13) ===================================


In [8]:
PROMPT_V1 = """Write a short social media post for a government announcement.
Announcement: {announcement}
Post: """

PROMPT_V2 = """You write social media posts for a government communications department.

Rules:
1. Keep posts under 280 characters.
2. Use a professional but approachable tone.
3. Include the hashtag #CityAlert at the end of every post.
4. Include one relevant emoji at the start of the post.
5. For emergencies, begin with ALERT:.

Announcement: {announcement}
Post: """

announcements = [
    "All public schools will be closed tomorrow due to a winter storm warning.",
    "City Hall will be closed on July 4th for Independence Day.",
    "A tornado warning has been issued for the county until 6pm tonight.",
    "Main Street will be closed for repaving from Monday through Wednesday.",
    "Free flu shots are available at the community center this weekend.",
]

# Build separate dataframes — each needs a column literally named 'prompt'
eval_df_v1 = pd.DataFrame({
    "prompt": [PROMPT_V1.format(announcement=ann) for ann in announcements],
    "context": announcements,
    "instruction": ["Generate a social media post following government communications guidelines."] * len(announcements),
})

eval_df_v2 = pd.DataFrame({
    "prompt": [PROMPT_V2.format(announcement=ann) for ann in announcements],
    "context": announcements,
    "instruction": ["Generate a social media post following government communications guidelines."] * len(announcements),
})

print(f"V1 dataset: {len(eval_df_v1)} rows")
print(f"V2 dataset: {len(eval_df_v2)} rows")
eval_df_v1.head(2)

V1 dataset: 5 rows
V2 dataset: 5 rows


,prompt,context,instruction
0,Write a short social media post for a governme...,All public schools will be closed tomorrow due...,Generate a social media post following governm...
1,Write a short social media post for a governme...,City Hall will be closed on July 4th for Indep...,Generate a social media post following governm...


In [9]:
from vertexai.evaluation import PointwiseMetric, PointwiseMetricPromptTemplate

coherence_metric = PointwiseMetric(
    metric="coherence",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "Coherence": "The post is logically organized, clear, and easy to understand as a standalone social media post."
        },
        rating_rubric={
            "5": "Completely coherent and easy to follow.",
            "4": "Mostly coherent with minor issues.",
            "3": "Somewhat coherent but noticeable gaps.",
            "2": "Difficult to follow.",
            "1": "Incoherent.",
        },
        input_variables=["prompt"],
    )
)

fluency_metric = PointwiseMetric(
    metric="fluency",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "Fluency": "The post uses grammatically correct, natural-sounding language appropriate for government social media."
        },
        rating_rubric={
            "5": "Perfectly fluent and natural.",
            "4": "Mostly fluent with minor awkwardness.",
            "3": "Somewhat fluent but noticeably unnatural.",
            "2": "Difficult to read due to language issues.",
            "1": "Not fluent at all.",
        },
        input_variables=["prompt"],
    )
)

rule_compliance_metric = PointwiseMetric(
    metric="rule_compliance",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "Rule Compliance": (
                "The post follows all required rules: "
                "(1) under 280 characters, "
                "(2) ends with #CityAlert, "
                "(3) starts with an emoji, "
                "(4) uses a professional but approachable tone."
            )
        },
        rating_rubric={
            "5": "Follows all rules perfectly.",
            "4": "Follows most rules with one minor violation.",
            "3": "Follows some rules but misses at least two.",
            "2": "Follows few rules.",
            "1": "Follows none of the rules.",
        },
        input_variables=["prompt"],
    )
)

eval_metrics = [coherence_metric, fluency_metric, rule_compliance_metric]
print("Metrics defined: coherence, fluency, rule_compliance")

Metrics defined: coherence, fluency, rule_compliance


In [10]:
eval_task_v1 = EvalTask(
    dataset=eval_df_v1,
    metrics=eval_metrics,
    experiment="social-post-prompt-comparison",
)

run_ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
result_v1 = eval_task_v1.evaluate(
    model=model,
    experiment_run_name=f"prompt-v1-baseline-{run_ts}",
)

print("=== Prompt V1 (Baseline) Summary Metrics ===")
print(result_v1.summary_metrics)

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'model_name': 'publishers/google/models/gemini-2.5-flash'}
INFO:vertexai.evaluation._evaluation:Generating a total of 5 responses from Gemini model gemini-2.5-flash.
100%|██████████| 5/5 [00:11<00:00,  2.35s/it]
INFO:vertexai.evaluation._evaluation:All 5 responses are successfully generated from Gemini model gemini-2.5-flash.
INFO:vertexai.evaluation._evaluation:Multithreaded Batch Inference took: 11.75736163700276 seconds.
INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 15 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 15/15 [00:57<00:00,  3.81s/it]
INFO:vertexai.evaluation._evaluation:All 15 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:57.12292640600208 seconds


=== Prompt V1 (Baseline) Summary Metrics ===
{'row_count': 5, 'coherence/mean': np.float64(5.0), 'coherence/std': 0.0, 'fluency/mean': np.float64(5.0), 'fluency/std': 0.0, 'rule_compliance/mean': np.float64(3.8), 'rule_compliance/std': 0.4472135954999579}


In [13]:
eval_task_v2 = EvalTask(
    dataset=eval_df_v2,
    metrics=eval_metrics,
    experiment="social-post-prompt-comparison",
)

run_ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
result_v2 = eval_task_v2.evaluate(
    model=model,
    experiment_run_name=f"prompt-v2-detailed-{run_ts}",
)

print("=== Prompt V2 (Detailed) Summary Metrics ===")
print(result_v2.summary_metrics)

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'model_name': 'publishers/google/models/gemini-2.5-flash'}
INFO:vertexai.evaluation._evaluation:Generating a total of 5 responses from Gemini model gemini-2.5-flash.
100%|██████████| 5/5 [00:03<00:00,  1.45it/s]
INFO:vertexai.evaluation._evaluation:All 5 responses are successfully generated from Gemini model gemini-2.5-flash.
INFO:vertexai.evaluation._evaluation:Multithreaded Batch Inference took: 3.4559416439988127 seconds.
INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 15 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 15/15 [00:31<00:00,  2.11s/it]
INFO:vertexai.evaluation._evaluation:All 15 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:31.728290545001073 seconds


=== Prompt V2 (Detailed) Summary Metrics ===
{'row_count': 5, 'coherence/mean': np.float64(5.0), 'coherence/std': 0.0, 'fluency/mean': np.float64(5.0), 'fluency/std': 0.0, 'rule_compliance/mean': np.float64(5.0), 'rule_compliance/std': 0.0}


In [16]:
def display_comparison(v1_result, v2_result, metric_names):
    print(f"{'Metric':<25} {'V1 Baseline':>14} {'V2 Detailed':>14} {'Delta':>10}")
    print("-" * 67)
    for name in metric_names:
        v1_val, v2_val = None, None
        for k, v in v1_result.summary_metrics.items():
            if name in k:
                v1_val = v
                v2_val = v2_result.summary_metrics.get(k, 0)
                break
        if v1_val is not None:
            delta = v2_val - v1_val
            marker = "▲" if delta > 0 else ("▼" if delta < 0 else "=")
            print(f"{name:<25} {v1_val:>14.3f} {v2_val:>14.3f} {marker}{abs(delta):>8.3f}")
        else:
            print(f"{name:<25} {'N/A':>14} {'N/A':>14}")
    print()
    print("▲ = V2 better   ▼ = V2 worse")

metric_names = ["coherence", "fluency", "rule_compliance"]
print("\n=== Prompt Comparison: V1 Baseline vs V2 Detailed ===")
display_comparison(result_v1, result_v2, metric_names)


=== Prompt Comparison: V1 Baseline vs V2 Detailed ===
Metric                       V1 Baseline    V2 Detailed      Delta
-------------------------------------------------------------------
coherence                          5.000          5.000 =   0.000
fluency                            5.000          5.000 =   0.000
rule_compliance                    3.800          5.000 ▲   1.200

▲ = V2 better   ▼ = V2 worse


In [20]:
def print_per_row_metrics(result, label):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    metric_names = ["coherence", "fluency", "rule_compliance"]
    for i, row in result.metrics_table.iterrows():
        # Shorten the prompt to just the announcement portion
        prompt_text = str(row.get("prompt", ""))
        # Extract just the announcement line
        for line in prompt_text.splitlines():
            if "Announcement:" in line:
                prompt_text = line.replace("Announcement:", "").strip()
                break
        else:
            prompt_text = prompt_text[:80] + "..."

        response_text = str(row.get("response", ""))[:120].replace("\n", " ")

        print(f"\n  [{i+1}] {prompt_text}")
        print(f"      Response : {response_text}...")
        print(f"      {'Metric':<20} {'Score':>6}  Explanation")
        print(f"      {'-'*70}")
        for m in metric_names:
            score = row.get(f"{m}/score", "N/A")
            explanation = str(row.get(f"{m}/explanation", ""))[:80]
            print(f"      {m:<20} {str(score):>6}  {explanation}")

print_per_row_metrics(result_v1, "Prompt V1 — Baseline")
print_per_row_metrics(result_v2, "Prompt V2 — Detailed")


  Prompt V1 — Baseline

  [1] All public schools will be closed tomorrow due to a winter storm warning.
      Response : Here are a few options, choose the one that best fits your government's usual tone:  ---  **Option 1 (Concise & Direct):...
      Metric                Score  Explanation
      ----------------------------------------------------------------------
      coherence               5.0  The response provides three excellent, coherent options for the social media pos
      fluency                 5.0  The AI-generated response provides multiple options, all of which are perfectly 
      rule_compliance         4.0  The AI provided multiple options instead of a single post as requested. Evaluati

  [2] City Hall will be closed on July 4th for Independence Day.
      Response : Here are a few options, pick the one that best fits your city's tone!  **Option 1 (Concise & Direct):**  🚨 **City Hall C...
      Metric                Score  Explanation
      ----------------------

Based on the findings here in the above code blocks - version 2 has a much better prompt that we would use if this was a real test.